In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
from pathlib import Path

import os, sys
sys.path.append(os.path.abspath("../src"))
from data_pipeline.config import PROCESSED_DIR, CLEANED, RAW_DIR


In [3]:
def flatten_cols(df):
    if isinstance(df.columns, pd.MultiIndex):
        if df.columns.get_level_values(1).nunique() == 1:
            df.columns = df.columns.get_level_values(0)
        else:
            df.columns = ["_".join(map(str, c)).strip("_") for c in df.columns]
    return df

In [4]:
# --- Realized Volatility (S&P 500; daily→monthly/quarterly RMS) ---
px = yf.download("^GSPC", start="1950-01-01", interval="1d", progress=False).dropna()
px.index = pd.to_datetime(px.index).tz_localize(None)

logret = np.log(px['Close']).diff()

display(px.head(3))
display(logret.head(3))

/var/folders/2p/b19f37nj6b98f824990xs99m0000gn/T/ipykernel_5387/3475955613.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  px = yf.download("^GSPC", start="1950-01-01", interval="1d", progress=False).dropna()


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
1950-01-03,16.66,16.66,16.66,16.66,1260000
1950-01-04,16.85,16.85,16.85,16.85,1890000
1950-01-05,16.93,16.93,16.93,16.93,2550000


Ticker,^GSPC
Date,
1950-01-03,NaN
1950-01-04,0.011340
1950-01-05,0.004737


In [5]:
spy = yf.download("SPY", start="1950-01-01", interval="1d", progress=False).dropna()
spy.index = pd.to_datetime(spy.index).tz_localize(None)

logret_spy = np.log(spy['Close']).diff()

print(spy.head(3))
print(logret_spy.head(3))

/var/folders/2p/b19f37nj6b98f824990xs99m0000gn/T/ipykernel_5387/2899969433.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy = yf.download("SPY", start="1950-01-01", interval="1d", progress=False).dropna()


Price           Close       High        Low       Open   Volume
Ticker            SPY        SPY        SPY        SPY      SPY
Date                                                           
1993-01-29  24.313044  24.330336  24.209289  24.330336  1003200
1993-02-01  24.485968  24.485968  24.330336  24.330336   480500
1993-02-02  24.537836  24.555128  24.416790  24.468667   201300
Ticker           SPY
Date                
1993-01-29       NaN
1993-02-01  0.007087
1993-02-02  0.002116


In [6]:
spy = flatten_cols(spy)
print(spy.columns)
print(spy.head(3))

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')
Price           Close       High        Low       Open   Volume
Date                                                           
1993-01-29  24.313044  24.330336  24.209289  24.330336  1003200
1993-02-01  24.485968  24.485968  24.330336  24.330336   480500
1993-02-02  24.537836  24.555128  24.416790  24.468667   201300


In [7]:
px = flatten_cols(px)
print(px.columns)
print(px.head(3))

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')
Price       Close   High    Low   Open   Volume
Date                                           
1950-01-03  16.66  16.66  16.66  16.66  1260000
1950-01-04  16.85  16.85  16.85  16.85  1890000
1950-01-05  16.93  16.93  16.93  16.93  2550000


In [8]:
# --- Realized volatility over next 21 days (annualized) ---
window = 21
rv_21d = np.sqrt(
    logret_spy.rolling(window, min_periods=window)
           .apply(lambda x: (x**2).sum(), raw=True)
           .shift(-window) * (252 / window)
)

rv_21d = rv_21d.rename(columns={"SPY": 'rv_21d'}) 
rv_21d = rv_21d.dropna()

rv_21d.head()

Ticker,rv_21d
Date,
1993-01-29,0.129208
1993-02-01,0.127671
1993-02-02,0.128905
1993-02-03,0.124028
1993-02-04,0.145034


In [9]:
# --- Realized volatility over next 21 days (annualized) ---
window = 60
rv_60d = np.sqrt(
    logret_spy.rolling(window, min_periods=window)
           .apply(lambda x: (x**2).sum(), raw=True)
           .shift(-window) * (252 / window)
)

rv_60d = rv_60d.rename(columns={"SPY": 'rv_60d'}) 
rv_60d = rv_60d.dropna()

rv_60d.head()

Ticker,rv_60d
Date,
1993-01-29,0.119567
1993-02-01,0.118763
1993-02-02,0.119006
1993-02-03,0.117075
1993-02-04,0.117489


In [10]:
# --- Realized volatility over next 21 days (annualized) ---
window = 120
rv_120d = np.sqrt(
    logret_spy.rolling(window, min_periods=window)
           .apply(lambda x: (x**2).sum(), raw=True)
           .shift(-window) * (252 / window)
)

rv_120d = rv_120d.rename(columns={"SPY": 'rv_120d'}) 
rv_120d = rv_120d.dropna()

rv_120d.head()

Ticker,rv_120d
Date,
1993-01-29,0.107819
1993-02-01,0.107563
1993-02-02,0.107823
1993-02-03,0.106745
1993-02-04,0.106617


In [11]:
spy = spy.rename(columns={
    'Open': 'open_spy', 'High': 'high_spy', 'Low': 'low_spy', 
    'Close': 'close_spy', 'Adj Close': 'adjclose_spy'
})
spx = px.rename(columns={
    'Open': 'open_gspc', 'High': 'high_gspc', 'Low': 'low_gspc', 
    'Close': 'close_gspc', 'Adj Close': 'adjclose_gspc'
})

spy = spy.drop(columns='Volume', errors='ignore')
spx  = spx.drop(columns='Volume', errors='ignore')

display(spy.head(3))
display(spx.head(3))

Price,close_spy,high_spy,low_spy,open_spy
Date,,,,
1993-01-29,24.313044,24.330336,24.209289,24.330336
1993-02-01,24.485968,24.485968,24.330336,24.330336
1993-02-02,24.537836,24.555128,24.416790,24.468667


Price,close_gspc,high_gspc,low_gspc,open_gspc
Date,,,,
1950-01-03,16.66,16.66,16.66,16.66
1950-01-04,16.85,16.85,16.85,16.85
1950-01-05,16.93,16.93,16.93,16.93


In [12]:
spx = spx.join(rv_21d, how='inner')
display(spx.head(3))

spy = spy.join(rv_21d, how='inner')
display(spy.head(3))

,close_gspc,high_gspc,low_gspc,open_gspc,rv_21d
Date,,,,,
1993-01-29,438.779999,438.929993,436.910004,438.670013,0.129208
1993-02-01,442.519989,442.519989,438.779999,438.779999,0.127671
1993-02-02,442.549988,442.869995,440.760010,442.519989,0.128905


,close_spy,high_spy,low_spy,open_spy,rv_21d
Date,,,,,
1993-01-29,24.313044,24.330336,24.209289,24.330336,0.129208
1993-02-01,24.485968,24.485968,24.330336,24.330336,0.127671
1993-02-02,24.537836,24.555128,24.416790,24.468667,0.128905


In [13]:
spx = spx.join(rv_60d, how='inner')
display(spx.head(3))

spy = spy.join(rv_60d, how='inner')
display(spy.head(3))

,close_gspc,high_gspc,low_gspc,open_gspc,rv_21d,rv_60d
Date,,,,,,
1993-01-29,438.779999,438.929993,436.910004,438.670013,0.129208,0.119567
1993-02-01,442.519989,442.519989,438.779999,438.779999,0.127671,0.118763
1993-02-02,442.549988,442.869995,440.760010,442.519989,0.128905,0.119006


,close_spy,high_spy,low_spy,open_spy,rv_21d,rv_60d
Date,,,,,,
1993-01-29,24.313044,24.330336,24.209289,24.330336,0.129208,0.119567
1993-02-01,24.485968,24.485968,24.330336,24.330336,0.127671,0.118763
1993-02-02,24.537836,24.555128,24.416790,24.468667,0.128905,0.119006


In [ ]:
spx = spx.join(rv_120d, how='inner')

spy = spy.join(rv_120d, how='inner')

In [15]:
spx = spx.reset_index().rename(columns={'Date': 'date'})
spy = spy.reset_index().rename(columns={'Date': 'date'})

In [16]:
display(spx.head(3))
display(spy.head(3))

,date,close_gspc,high_gspc,low_gspc,open_gspc,rv_21d,rv_60d,rv_120d
0,1993-01-29,438.779999,438.929993,436.910004,438.670013,0.129208,0.119567,0.107819
1,1993-02-01,442.519989,442.519989,438.779999,438.779999,0.127671,0.118763,0.107563
2,1993-02-02,442.549988,442.869995,440.760010,442.519989,0.128905,0.119006,0.107823


,date,close_spy,high_spy,low_spy,open_spy,rv_21d,rv_60d,rv_120d
0,1993-01-29,24.313044,24.330336,24.209289,24.330336,0.129208,0.119567,0.107819
1,1993-02-01,24.485968,24.485968,24.330336,24.330336,0.127671,0.118763,0.107563
2,1993-02-02,24.537836,24.555128,24.416790,24.468667,0.128905,0.119006,0.107823


In [17]:
spy.to_csv(RAW_DIR / "SPY_data.csv", index=False)
spx.to_csv(RAW_DIR / "GSPC_data.csv", index=False)